# Pratica — Lezione 4: Addestrare il modello

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife.

En la clase completaste un bucle de entrenamiento entero sobre MNIST: función de pérdida, optimizador y las cuatro líneas del ciclo (`zero_grad`, forward, `backward`, `step`). En esta práctica vas a hacerlo tú mismo. Hay dos ejercicios:

1. Completar los huecos de un bucle de entrenamiento ya montado
2. Cambiar de optimizador y comparar cuántas épocas hacen falta para llegar a una pérdida parecida

Cada ejercicio tiene una celda de código con el andamiaje ya preparado y partes marcadas con `# TODO` para que las completes tú. Intenta resolverlas antes de mirar la solución: aprenderás mucho más equivocándote y corrigiendo que leyendo directamente la respuesta.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

print("Versión de PyTorch:", torch.__version__)

# Datos: los mismos de la clase, en lotes de 64 imágenes
transformaciones = transforms.Compose([transforms.ToTensor()])

datos_entrenamiento = torchvision.datasets.MNIST(
    root="./datos", train=True, download=True, transform=transformaciones
)
datos_prueba = torchvision.datasets.MNIST(
    root="./datos", train=False, download=True, transform=transformaciones
)

cargador_entrenamiento = torch.utils.data.DataLoader(datos_entrenamiento, batch_size=64, shuffle=True)
cargador_prueba = torch.utils.data.DataLoader(datos_prueba, batch_size=64, shuffle=False)


# Red: la misma arquitectura de la clase 3 y la clase 4
class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(28 * 28, 128)
        self.capa2 = nn.Linear(128, 64)
        self.capa3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.capa1(x))
        x = F.relu(self.capa2(x))
        x = self.capa3(x)
        return x


print("Datos y arquitectura listos.")

## Esercizio 1 — Completa il ciclo di addestramento

El andamiaje de abajo ya crea la red, la función de pérdida y el optimizador, y ya tiene montado el bucle de dos niveles (épocas y lotes). Te faltan los cuatro pasos del ciclo de entrenamiento que viste en la clase.

Pistas:

- El primer paso del ciclo reinicia los gradientes acumulados del optimizador en el paso anterior
- El segundo paso es el forward: pasa `imagenes` por `red` y guarda el resultado
- El tercer paso calcula la pérdida comparando esa salida con `etiquetas`, usando `funcion_perdida`
- El cuarto paso (`backward`) calcula los gradientes a partir de la pérdida
- El quinto paso (`step`) le pide al optimizador que actualice los pesos con esos gradientes
- Recuerda el orden: reiniciar gradientes, forward, loss, backward, step

In [ ]:
red_ejercicio1 = RedMNIST()
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(red_ejercicio1.parameters(), lr=0.001)

epocas = 3
for epoca in range(epocas):
    for imagenes, etiquetas in cargador_entrenamiento:
        # TODO: reinicia los gradientes acumulados del optimizador

        # TODO: forward, guarda la predicción de la red en una variable llamada "salida"

        # TODO: calcula la pérdida comparando "salida" con "etiquetas" usando funcion_perdida,
        # guárdala en una variable llamada "perdida"

        # TODO: backward, calcula los gradientes a partir de "perdida"

        # TODO: pide al optimizador que actualice los pesos
        pass

    print(f"Época {epoca + 1}/{epocas} completada")

<details>
<summary>💡 Vedi soluzione</summary>

```python
red_ejercicio1 = RedMNIST()
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(red_ejercicio1.parameters(), lr=0.001)

epocas = 3
for epoca in range(epocas):
    perdida_epoca = 0.0
    for imagenes, etiquetas in cargador_entrenamiento:
        optimizador.zero_grad()                       # reinicia los gradientes del paso anterior

        salida = red_ejercicio1(imagenes)              # forward: la red genera una predicción

        perdida = funcion_perdida(salida, etiquetas)    # loss: cuánto se equivocó la red

        perdida.backward()                              # backward: calcula el gradiente de cada peso

        optimizador.step()                              # optimizer.step: actualiza los pesos

        perdida_epoca += perdida.item()

    print(f"Época {epoca + 1}/{epocas} - pérdida media: {perdida_epoca / len(cargador_entrenamiento):.4f}")
```
</details>

## Esercizio 2 — SGD vs Adam

En la clase entrenaste con `Adam`. Ahora vas a entrenar la misma red, con los mismos datos, usando `SGD` (Stochastic Gradient Descent, descenso de gradiente estocástico) en su lugar, y vas a comparar cuántas épocas hacen falta con cada optimizador para llegar a una pérdida media parecida.

Pistas:

- Crea una red nueva (`RedMNIST()`) para no arrancar desde pesos ya entrenados en el ejercicio 1
- `optim.SGD` necesita los mismos dos argumentos que `optim.Adam`: los parámetros de la red y la tasa de aprendizaje (`lr`)
- Con `SGD` sin más ajustes conviene una tasa de aprendizaje más alta que con Adam; prueba con `lr=0.1`
- Entrena durante el mismo número de épocas que en el ejercicio 1 (tres) y compara la pérdida media final con la que obtuviste con Adam
- Si con tres épocas la pérdida de `SGD` queda claramente por encima de la de Adam, prueba a aumentar el número de épocas hasta que se acerquen, y anota cuántas hicieron falta

In [ ]:
# TODO: crea una red nueva para este ejercicio
red_ejercicio2 = None

funcion_perdida = nn.CrossEntropyLoss()

# TODO: crea un optimizador SGD sobre red_ejercicio2.parameters(), con lr=0.1
optimizador_sgd = None

epocas = 3
for epoca in range(epocas):
    perdida_epoca = 0.0
    for imagenes, etiquetas in cargador_entrenamiento:
        # TODO: repite aquí las cinco líneas del ciclo de entrenamiento del ejercicio 1,
        # usando red_ejercicio2 y optimizador_sgd
        pass

    print(f"Época {epoca + 1}/{epocas} - pérdida media (SGD): {perdida_epoca / len(cargador_entrenamiento):.4f}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
red_ejercicio2 = RedMNIST()
funcion_perdida = nn.CrossEntropyLoss()
optimizador_sgd = optim.SGD(red_ejercicio2.parameters(), lr=0.1)

epocas = 3
for epoca in range(epocas):
    perdida_epoca = 0.0
    for imagenes, etiquetas in cargador_entrenamiento:
        optimizador_sgd.zero_grad()
        salida = red_ejercicio2(imagenes)
        perdida = funcion_perdida(salida, etiquetas)
        perdida.backward()
        optimizador_sgd.step()
        perdida_epoca += perdida.item()

    print(f"Época {epoca + 1}/{epocas} - pérdida media (SGD): {perdida_epoca / len(cargador_entrenamiento):.4f}")

# Con lr=0.1, SGD suele necesitar más de tres épocas para acercarse a la pérdida
# que Adam alcanza en tres. Sube "epocas" y compara: es habitual que Adam converja
# más rápido en las primeras épocas porque adapta el tamaño de paso por cada peso,
# mientras que SGD usa un tamaño de paso fijo para todos.
```
</details>

## Per continuare a esercitarti

Se ti è rimasto tempo, aggiungi il calcolo della precisione (accuracy) su `cargador_prueba` alla fine di ogni esercizio, come hai fatto nel notebook di teoria, e verifica se una perdita di addestramento più bassa si traduce sempre in una precisione di test più alta. Puoi anche provare altri tassi di apprendimento per `SGD` (ad esempio, `lr=0.01` o `lr=0.5`) e osservare come cambia la velocità di convergenza.